<a href="https://colab.research.google.com/github/AidoruFusion/northstar-databases-analytics/blob/main/MongoDB_Notebook_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 02: My MongoDB Atlas Development Workflow

This notebook documents how I used MongoDB Atlas and PyMongo to build the NoSQL database component for the NorthStar case study.

My workflow in this notebook is:

- connect Google Colab to MongoDB Atlas securely;
- load the cleaned JSON files produced from Notebook 01;
- upload each cleaned dataset into a MongoDB collection;
- remodel selected datasets into an integrated `NorthStar_Service_Cases` collection;
- demonstrate CRUD operations;
- run aggregation pipelines to investigate NorthStar's operational issues;
- create indexes and use `explain` to check query performance.


## Preparing the MongoDB notebook environment

In this section, I import the Python packages needed for the MongoDB workflow.

The packages are used to:

- connect Python to MongoDB Atlas using `pymongo`;
- check and prepare data using `pandas`;
- load cleaned JSON files using `json`;
- manage project folders using `pathlib`;
- keep the MongoDB URI hidden during runtime.


In [8]:
!pip install pymongo dnspython -q

import json
import pandas as pd
from pathlib import Path
from pymongo import MongoClient, ASCENDING, DESCENDING
from pymongo.errors import ServerSelectionTimeoutError, OperationFailure

## Connecting my notebook to MongoDB Atlas

In this section, I connect Google Colab to MongoDB Atlas using PyMongo.

The MongoDB URI is left blank in the notebook and entered at runtime so that my database credentials are not saved into GitHub.


In [9]:
from getpass import getpass

MONGO_URI = getpass("MongoDB Atlas URI: ")

DATABASE_NAME = "North-Database"

if MONGO_URI.strip() == "":
    print("MongoDB URI is blank. Paste your connection string when prompted before running connection cells.")
else:
    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    client.admin.command("ping")
    db = client[DATABASE_NAME]
    print("Connected successfully to MongoDB Atlas.")


MongoDB Atlas URI: ··········
Connected successfully to MongoDB Atlas.


## Locating my cleaned JSON files

In this section, I point the notebook to the cleaned JSON files created after the Python cleaning stage.

The expected GitHub project folder is:

"data/cleaned/json/"

This keeps my workflow clear:

raw CSV files → cleaned CSV files → cleaned JSON files → MongoDB Atlas collections.


In [10]:
from pathlib import Path

repo_url = "https://github.com/AidoruFusion/northstar-databases-analytics.git"
repo_folder = Path("/content/northstar-databases-analytics")

if repo_folder.exists():
    %cd /content/northstar-databases-analytics
    !git pull origin main
else:
    %cd /content
    !git clone {repo_url}

print("Repository ready.")

/content/northstar-databases-analytics
From https://github.com/AidoruFusion/northstar-databases-analytics
 * branch            main       -> FETCH_HEAD
Already up to date.
Repository ready.


## Mapping cleaned JSON files to MongoDB collections

In this section, I define how each cleaned JSON file will be stored as a MongoDB collection.

This gives each dataset a clear collection name in the `North-Database` database.


In [11]:
collection_map = {
    "customers.json": "Clean_Customers",
    "orders.json": "Cleaned_Orders",
    "deliveries.json": "Cleaned_Deliveries",
    "drivers.json": "Clean_Driver",
    "vehicles.json": "Cleaned_Vehicles",
    "hubs.json": "Clean_Hubs",
    "complaints.json": "Clean_Complaints",
    "incidents.json": "Cleaned_Incidents",
    "app_events.json": "Clean_app_events",
    "data_dictionary.json": "Cleaned_data_dictionary"
}

collection_map

{'customers.json': 'Clean_Customers',
 'orders.json': 'Cleaned_Orders',
 'deliveries.json': 'Cleaned_Deliveries',
 'drivers.json': 'Clean_Driver',
 'vehicles.json': 'Cleaned_Vehicles',
 'hubs.json': 'Clean_Hubs',
 'complaints.json': 'Clean_Complaints',
 'incidents.json': 'Cleaned_Incidents',
 'app_events.json': 'Clean_app_events',
 'data_dictionary.json': 'Cleaned_data_dictionary'}

## Uploading my cleaned JSON files into MongoDB Atlas

In this section, I read each cleaned JSON file and insert it into its matching MongoDB collection.

Each collection is cleared before insertion so that rerunning the notebook does not create duplicate records.


In [12]:
def load_json_collection(json_path, collection_name):
    with open(json_path, "r", encoding="utf-8") as file:
        records = json.load(file)

    collection = db[collection_name]
    collection.delete_many({})

    if len(records) > 0:
        collection.insert_many(records)

    print(f"{collection_name}: inserted {len(records)} records")


if MONGO_URI != "":
    json_folder = repo_folder / "data/cleaned/json/"

    for json_file, collection_name in collection_map.items():
        json_path = json_folder / json_file

        if json_path.exists():
            load_json_collection(json_path, collection_name)
        else:
            print(f"Missing file: {json_file}")

Clean_Customers: inserted 650 records
Cleaned_Orders: inserted 1250 records
Cleaned_Deliveries: inserted 950 records
Clean_Driver: inserted 170 records
Cleaned_Vehicles: inserted 120 records
Clean_Hubs: inserted 8 records
Clean_Complaints: inserted 320 records
Cleaned_Incidents: inserted 280 records
Clean_app_events: inserted 640 records
Cleaned_data_dictionary: inserted 9 records


## Checking that my MongoDB collections uploaded correctly

In this section, I check the document count for each MongoDB collection.

This confirms that the cleaned JSON files were successfully transferred into MongoDB Atlas.


In [13]:
if MONGO_URI != "":
    for collection_name in collection_map.values():
        count = db[collection_name].count_documents({})
        print(f"{collection_name}: {count} documents")

Clean_Customers: 650 documents
Cleaned_Orders: 1250 documents
Cleaned_Deliveries: 950 documents
Clean_Driver: 170 documents
Cleaned_Vehicles: 120 documents
Clean_Hubs: 8 documents
Clean_Complaints: 320 documents
Cleaned_Incidents: 280 documents
Clean_app_events: 640 documents
Cleaned_data_dictionary: 9 documents


# My NoSQL Design Rationale

NorthStar has fragmented operational data across customers, orders, deliveries, complaints, incidents, vehicles, drivers, hubs, and app events.

My MongoDB design uses two layers:

- separate collections to preserve each cleaned source dataset;
- an integrated `NorthStar_Service_Cases` collection to connect related customer, order, delivery, complaint, incident, and app-event records in one document.

This design matches the case study because NorthStar needs a more flexible document-based database that can connect fragmented service records and support operational investigation.


## Remodelling selected files into an integrated NorthStar service case collection

In this section, I remodel selected datasets into a document-based `NorthStar_Service_Cases` collection.

Each service-case document can contain:

- order details;
- customer details;
- delivery outcome;
- related complaints;
- related incidents;
- related app events.

This shows how MongoDB can hold related operational information together instead of keeping every record completely separate.


In [14]:
def create_northstar_service_cases():
    orders = list(db["Cleaned_Orders"].find({}, {"_id": 0}))
    customers = {c['customer_id']: c for c in db["Clean_Customers"].find({}, {"_id": 0})}
    deliveries = {d['order_id']: d for d in db["Cleaned_Deliveries"].find({}, {"_id": 0})}

    complaints_map = {}
    for c in db["Clean_Complaints"].find({}, {"_id": 0}):
        complaints_map.setdefault(c.get("order_id"), []).append(c)

    incidents_map = {}
    for i in db["Cleaned_Incidents"].find({}, {"_id": 0}):
        incidents_map.setdefault(i.get("delivery_id"), []).append(i)

    events_map = {}
    for e in db["Clean_app_events"].find({}, {"_id": 0}):
        events_map.setdefault(e.get("order_id"), []).append(e)

    cases = []
    for order in orders:
        order_id = order.get("order_id")
        customer_id = order.get("customer_id")

        customer = customers.get(customer_id)
        delivery = deliveries.get(order_id)
        complaints = complaints_map.get(order_id, [])

        incidents = []
        if delivery and delivery.get("delivery_id"):
            incidents = incidents_map.get(delivery.get("delivery_id"), [])

        app_events = events_map.get(order_id, [])

        case_document = {
            "case_id": f"CASE_{order_id}",
            "order_id": order_id,
            "customer_id": customer_id,
            "customer": customer,
            "order": order,
            "delivery": delivery,
            "complaints": complaints,
            "incidents": incidents,
            "app_events": app_events,
            "complaint_count": len(complaints),
            "incident_count": len(incidents),
            "app_event_count": len(app_events)
        }
        cases.append(case_document)

    db["NorthStar_Service_Cases"].delete_many({})
    if cases:
        db["NorthStar_Service_Cases"].insert_many(cases)

    print(f"NorthStar_Service_Cases inserted: {len(cases)}")

if MONGO_URI != "":
    create_northstar_service_cases()

NorthStar_Service_Cases inserted: 1250


## Checking one integrated NorthStar service service-case document

In this section, I preview one document from `NorthStar_Service_Cases`.

This helps confirm that the remodelling created the intended nested document structure.


In [15]:
if MONGO_URI != "":
    sample_case = db["NorthStar_Service_Cases"].find_one({}, {"_id": 0})
    sample_case
else:
    print("Skipping preview because MONGO_URI is blank.")

# Demonstrating CRUD Operations

In this section, I demonstrate the four core MongoDB CRUD operations using PyMongo:

- Create;
- Read;
- Update;
- Delete.

I use a small demo service review record so that the CRUD workflow can be tested safely without changing the main NorthStar collections.


## CRUD Create: inserting a demo service review

In this section, I insert one demo service review document.

The record is clearly marked as a demo record so it can be updated and deleted later in the workflow.


In [17]:
if MONGO_URI != "":
    demo_record = {
        "case_id": "DEMO_CASE_001",
        "order_id": "DEMO_ORDER_001",
        "customer_id": "DEMO_CUSTOMER_001",
        "issue_type": "Late Arrival",
        "zone": "Central",
        "status": "Open",
        "priority": "High",
        "notes": "Demo CRUD record created for coursework testing."
    }

    result = db["Service_Review_Demo"].insert_one(demo_record)
    print("Inserted demo document ID:", result.inserted_id)

Inserted demo document ID: 6a01c54bbe67b3d25b7b2175


## CRUD Read: querying documents from MongoDB

In this section, I use read queries to retrieve documents from MongoDB.

The queries demonstrate filtering, projection, sorting, and limiting results.


In [18]:
if MONGO_URI != "":
    print("Customers in Central zone:")
    for doc in db["Clean_Customers"].find(
        {"home_zone": "Central"},
        {"_id": 0, "customer_id": 1, "home_zone": 1, "customer_type": 1, "loyalty_score": 1}
    ).limit(5):
        print(doc)

    print("\nHigh-priority orders:")
    for doc in db["Cleaned_Orders"].find(
        {"priority_level": {"$in": ["High", "Urgent"]}},
        {"_id": 0, "order_id": 1, "customer_id": 1, "pickup_zone": 1, "priority_level": 1, "order_value": 1}
    ).sort("order_value", DESCENDING).limit(5):
        print(doc)
else:
    print("Skipping read operations because MONGO_URI is blank.")

Customers in Central zone:
{'customer_id': 'C0004', 'home_zone': 'Central', 'customer_type': 'Consumer', 'loyalty_score': 32.5}
{'customer_id': 'C0011', 'home_zone': 'Central', 'customer_type': 'Consumer', 'loyalty_score': 79.9}
{'customer_id': 'C0021', 'home_zone': 'Central', 'customer_type': 'SME', 'loyalty_score': 63.6}
{'customer_id': 'C0028', 'home_zone': 'Central', 'customer_type': 'Consumer', 'loyalty_score': 78.5}
{'customer_id': 'C0029', 'home_zone': 'Central', 'customer_type': 'Consumer', 'loyalty_score': 57.0}

High-priority orders:
{'order_id': 'O00144', 'customer_id': 'C0063', 'pickup_zone': 'Riverside', 'priority_level': 'High', 'order_value': 288.86}
{'order_id': 'O00790', 'customer_id': 'C0028', 'pickup_zone': 'Airport', 'priority_level': 'High', 'order_value': 283.08}
{'order_id': 'O00799', 'customer_id': 'C0289', 'pickup_zone': 'East', 'priority_level': 'High', 'order_value': 260.0}
{'order_id': 'O00115', 'customer_id': 'C0157', 'pickup_zone': 'East', 'priority_level'

## CRUD Update: modifying the demo record

In this section, I update the demo service review record.

This shows how MongoDB can change an existing document using PyMongo.


In [20]:
if MONGO_URI != "":
    result = db["Service_Review_Demo"].update_one(
            {"case_id": "DEMO_CASE_001"},
            {"$set": {"status": "Investigating", "assigned_team": "Operations"}}
        )

    print("Documents matched:", result.matched_count)
    print("Documents modified:", result.modified_count)

    updated_doc = db["Service_Review_Demo"].find_one({"case_id": "DEMO_CASE_001"}, {"_id": 0})
    print(updated_doc)

Documents matched: 1
Documents modified: 1
{'case_id': 'DEMO_CASE_001', 'order_id': 'DEMO_ORDER_001', 'customer_id': 'DEMO_CUSTOMER_001', 'issue_type': 'Late Arrival', 'zone': 'Central', 'status': 'Investigating', 'priority': 'High', 'notes': 'Demo CRUD record created for coursework testing.', 'assigned_team': 'Operations'}


## CRUD Delete: removing the demo record

In this section, I delete the demo service review record after testing.

This keeps the database clean after the CRUD demonstration.


In [21]:
if MONGO_URI != "":
  result = db["Service_Review_Demo"].delete_one({"case_id": "DEMO_CASE_001"})
  print("Documents deleted:", result.deleted_count)

Documents deleted: 1


# Running MongoDB Aggregation Pipelines

In this section, I use aggregation pipelines to analyse NorthStar's operational data.

The pipelines group, filter, join, and summarise records to identify patterns linked to service performance, complaints, delays, hubs, and incidents.


## Aggregation 1: order value by service type and zone

In this query, I group orders by service type and pickup zone.

This helps identify which service areas and locations generate the highest order value.


In [23]:
if MONGO_URI != "":
    pipeline = [
        {"$group": {
            "_id": {"service_type": "$service_type", "pickup_zone": "$pickup_zone"},
            "total_order_value": {"$sum": "$order_value"},
            "average_order_value": {"$avg": "$order_value"},
            "order_count": {"$sum": 1}
        }},
        {"$sort": {"total_order_value": -1}},
        {"$limit": 10}
    ]

    results = list(db["Cleaned_Orders"].aggregate(pipeline))
    display(pd.DataFrame(results))

,_id,total_order_value,average_order_value,order_count
0,"{'service_type': 'Passenger', 'pickup_zone': '...",5892.65,107.139091,55
1,"{'service_type': 'Passenger', 'pickup_zone': '...",5678.06,84.747164,67
2,"{'service_type': 'Retail', 'pickup_zone': 'Cen...",5441.86,83.720923,65
3,"{'service_type': 'Passenger', 'pickup_zone': '...",4891.96,94.076154,52
4,"{'service_type': 'Parcel', 'pickup_zone': 'East'}",4833.41,92.950192,52
5,"{'service_type': 'Parcel', 'pickup_zone': 'Cen...",4647.96,84.508364,55
6,"{'service_type': 'Passenger', 'pickup_zone': '...",4467.14,95.045532,47
7,"{'service_type': 'Passenger', 'pickup_zone': '...",4244.66,108.837436,39
8,"{'service_type': 'Passenger', 'pickup_zone': '...",4189.70,99.754762,42
9,"{'service_type': 'Retail', 'pickup_zone': 'Sou...",4133.59,98.418810,42


## Aggregation 2: delivery performance by hub

In this query, I analyse delivery performance by hub.

This supports the case study issue of underperforming hubs and service delays.


In [24]:
if MONGO_URI != "":
    pipeline = [
        {"$group": {
            "_id": "$hub_id",
            "delivery_count": {"$sum": 1},
            "avg_duration_hours": {"$avg": "$delivery_duration_hours"},
            "avg_rating": {"$avg": "$customer_rating_post_delivery"},
            "avg_cost": {"$avg": "$fuel_or_charge_cost"},
            "manual_override_total": {"$sum": "$manual_route_override_count"}
        }},
        {"$sort": {"avg_duration_hours": -1}}
    ]

    results = list(db["Cleaned_Deliveries"].aggregate(pipeline))
    display(pd.DataFrame(results).head(10))

,_id,delivery_count,avg_duration_hours,avg_rating,avg_cost,manual_override_total
0,H05,115,11.553300,3.669558,13.686000,109
1,H04,127,11.102635,3.915476,13.167008,111
2,H01,136,10.684968,3.840593,12.755809,140
3,H08,128,10.560490,3.884560,11.708203,142
4,H07,115,10.535888,3.881858,12.922087,121
5,H06,104,9.917423,3.882136,13.319231,95
6,H02,106,9.478948,3.950952,12.565000,97
7,H03,119,8.437893,3.895862,12.744202,106


## Aggregation 3: complaints by type and severity

In this query, I group complaints by type and severity.

This helps identify the main causes of customer dissatisfaction.


In [25]:
if MONGO_URI != "":
    pipeline = [
        {"$group": {
            "_id": {"complaint_type": "$complaint_type", "severity": "$severity"},
            "complaint_count": {"$sum": 1},
            "avg_resolution_days": {"$avg": "$resolution_days"},
            "total_compensation": {"$sum": "$compensation_amount"}
        }},
        {"$sort": {"complaint_count": -1}}
    ]

    results = list(db["Clean_Complaints"].aggregate(pipeline))
    display(pd.DataFrame(results).head(10))

,_id,complaint_count,avg_resolution_days,total_compensation
0,"{'complaint_type': 'Delay', 'severity': 'Medium'}",56,5.964286,964.87
1,"{'complaint_type': 'MissedPickup', 'severity':...",37,6.162162,644.80
2,"{'complaint_type': 'DriverBehaviour', 'severit...",31,5.419355,476.29
3,"{'complaint_type': 'Delay', 'severity': 'Low'}",27,6.481481,220.43
4,"{'complaint_type': 'AppIssue', 'severity': 'Me...",25,7.360000,386.58
5,"{'complaint_type': 'Delay', 'severity': 'High'}",18,12.444444,511.54
6,"{'complaint_type': 'DriverBehaviour', 'severit...",16,13.750000,460.63
7,"{'complaint_type': 'MissedPickup', 'severity':...",16,11.562500,689.11
8,"{'complaint_type': 'AppIssue', 'severity': 'Low'}",15,6.066667,185.55
9,"{'complaint_type': 'AppIssue', 'severity': 'Hi...",13,13.923077,408.59


## Aggregation 4: joining deliveries with hubs using `$lookup`

In this query, I use `$lookup` to join delivery records with hub information.

This demonstrates how MongoDB can connect related collections when analysis requires data from more than one source.


In [26]:
if MONGO_URI != "":
    pipeline = [
        {"$lookup": {
            "from": "Clean_Hubs",
            "localField": "hub_id",
            "foreignField": "hub_id",
            "as": "hub_info"
        }},
        {"$unwind": "$hub_info"},
        {"$group": {
            "_id": "$hub_info.zone",
            "delivery_count": {"$sum": 1},
            "avg_delivery_duration": {"$avg": "$delivery_duration_hours"},
            "avg_customer_rating": {"$avg": "$customer_rating_post_delivery"},
            "avg_cost": {"$avg": "$fuel_or_charge_cost"}
        }},
        {"$sort": {"avg_delivery_duration": -1}}
    ]

    results = list(db["Cleaned_Deliveries"].aggregate(pipeline))
    display(pd.DataFrame(results))

,_id,delivery_count,avg_delivery_duration,avg_customer_rating,avg_cost
0,West,127,11.102635,3.915476,13.167008
1,Central,243,11.034930,3.782479,12.644198
2,North,136,10.684968,3.840593,12.755809
3,Riverside,115,10.535888,3.881858,12.922087
4,Airport,104,9.917423,3.882136,13.319231
5,South,106,9.478948,3.950952,12.565000
6,East,119,8.437893,3.895862,12.744202


## Aggregation 5: identifying NorthStar service cases with complaints or incidents

In this query, I use the integrated `NorthStar_Service_Cases` collection to find cases with service problems.

This supports NorthStar's need to connect complaints, incidents, deliveries, and customer records in one operational view.


In [27]:
if MONGO_URI != "":
    pipeline = [
        {"$match": {"$or": [{"complaint_count": {"$gt": 0}}, {"incident_count": {"$gt": 0}}]}},
        {"$project": {
            "_id": 0,
            "case_id": 1,
            "order_id": 1,
            "customer_id": 1,
            "complaint_count": 1,
            "incident_count": 1,
            "app_event_count": 1,
            "pickup_zone": "$order.pickup_zone",
            "service_type": "$order.service_type",
            "delivery_status": "$delivery.delivery_status",
            "delivery_duration_hours": "$delivery.delivery_duration_hours"
        }},
        {"$sort": {"complaint_count": -1, "incident_count": -1}},
        {"$limit": 10}
    ]

    results = list(db["NorthStar_Service_Cases"].aggregate(pipeline))
    display(pd.DataFrame(results))

,case_id,order_id,customer_id,complaint_count,incident_count,app_event_count,pickup_zone,service_type,delivery_status,delivery_duration_hours
0,CASE_O00795,O00795,C0142,3,0,0,Riverside,Retail,NaN,NaN
1,CASE_O00125,O00125,C0191,3,0,0,West,Retail,NaN,NaN
2,CASE_O01151,O01151,C0100,2,2,0,South,Retail,OnTime,12.478753
3,CASE_O00628,O00628,C0056,2,1,1,Riverside,Passenger,Delayed,15.098799
4,CASE_O01104,O01104,C0480,2,1,0,East,Parcel,OnTime,11.079243
5,CASE_O00483,O00483,C0285,2,1,0,Riverside,Business,Failed,14.982770
6,CASE_O00417,O00417,C0486,2,1,0,South,Retail,OnTime,3.717208
7,CASE_O00443,O00443,C0378,2,1,0,North,Parcel,OnTime,8.984128
8,CASE_O00351,O00351,C0259,2,1,0,Airport,Parcel,OnTime,0.640782
9,CASE_O00270,O00270,C0561,2,0,1,West,Business,OnTime,0.963972


# Applying Indexing and Query Optimisation

In this section, I create indexes and test query performance.

Indexes are important because they help MongoDB search collections more efficiently instead of scanning every document.


## Creating indexes for frequently queried fields

In this section, I create indexes on fields that are likely to be used in operational queries.

The indexes support faster searching for customers, orders, delivery status, complaints, app events, and integrated NorthStar service cases.


In [28]:
if MONGO_URI != "":
    db["Clean_Customers"].create_index([("customer_id", ASCENDING)], name="idx_customer_id")
    db["Clean_Customers"].create_index([("home_zone", ASCENDING)], name="idx_home_zone")

    db["Cleaned_Orders"].create_index([("order_id", ASCENDING)], name="idx_order_id")
    db["Cleaned_Orders"].create_index([("customer_id", ASCENDING)], name="idx_order_customer_id")
    db["Cleaned_Orders"].create_index([("pickup_zone", ASCENDING), ("service_type", ASCENDING)], name="idx_zone_service")

    db["Cleaned_Deliveries"].create_index([("delivery_id", ASCENDING)], name="idx_delivery_id")
    db["Cleaned_Deliveries"].create_index([("order_id", ASCENDING)], name="idx_delivery_order_id")
    db["Cleaned_Deliveries"].create_index([("hub_id", ASCENDING), ("delivery_status", ASCENDING)], name="idx_hub_status")

    db["Clean_Complaints"].create_index([("order_id", ASCENDING)], name="idx_complaint_order_id")
    db["Clean_Complaints"].create_index([("complaint_type", ASCENDING), ("severity", ASCENDING)], name="idx_type_severity")

    db["Cleaned_Incidents"].create_index([("delivery_id", ASCENDING)], name="idx_incident_delivery_id")
    db["Clean_app_events"].create_index([("order_id", ASCENDING)], name="idx_event_order_id")
    db["Clean_app_events"].create_index([("zone_context", ASCENDING)], name="idx_event_zone")

    db["NorthStar_Service_Cases"].create_index([("order_id", ASCENDING)], name="idx_case_order_id")
    db["NorthStar_Service_Cases"].create_index([("customer_id", ASCENDING)], name="idx_case_customer_id")
    db["NorthStar_Service_Cases"].create_index([("complaint_count", DESCENDING), ("incident_count", DESCENDING)], name="idx_case_problem_counts")

    print("Indexes created successfully.")

Indexes created successfully.


## Checking the indexes created in MongoDB

In this section, I display the indexes for the relevant collections.

This confirms that the indexing stage has been completed.


In [29]:
if MONGO_URI != "":
    for collection_name in ["Clean_Customers", "Cleaned_Orders", "Cleaned_Deliveries", "Clean_Complaints", "NorthStar_Service_Cases"]:
        print("\nIndexes for:", collection_name)
        for index in db[collection_name].list_indexes():
            print(index)


Indexes for: Clean_Customers
SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')])
SON([('v', 2), ('key', SON([('customer_id', 1)])), ('name', 'idx_customer_id')])
SON([('v', 2), ('key', SON([('home_zone', 1)])), ('name', 'idx_home_zone')])

Indexes for: Cleaned_Orders
SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')])
SON([('v', 2), ('key', SON([('order_id', 1)])), ('name', 'idx_order_id')])
SON([('v', 2), ('key', SON([('customer_id', 1)])), ('name', 'idx_order_customer_id')])
SON([('v', 2), ('key', SON([('pickup_zone', 1), ('service_type', 1)])), ('name', 'idx_zone_service')])

Indexes for: Cleaned_Deliveries
SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')])
SON([('v', 2), ('key', SON([('delivery_id', 1)])), ('name', 'idx_delivery_id')])
SON([('v', 2), ('key', SON([('order_id', 1)])), ('name', 'idx_delivery_order_id')])
SON([('v', 2), ('key', SON([('hub_id', 1), ('delivery_status', 1)])), ('name', 'idx_hub_status')])

Indexes for: Clean_Complaints
SON(

## Testing query performance with `explain`

In this section, I use MongoDB's `explain` command to inspect the query plan.

The aim is to check whether MongoDB uses an index instead of scanning the full collection.


In [30]:
def explain_find(collection_name, filter_query):
    explain_result = db.command(
        "explain",
        {"find": collection_name, "filter": filter_query},
        verbosity="executionStats"
    )

    execution_stats = explain_result.get("executionStats", {})
    query_planner = explain_result.get("queryPlanner", {})

    print("Collection:", collection_name)
    print("Filter:", filter_query)
    print("Documents returned:", execution_stats.get("nReturned"))
    print("Documents examined:", execution_stats.get("totalDocsExamined"))
    print("Execution time ms:", execution_stats.get("executionTimeMillis"))
    print("Winning plan:")
    print(query_planner.get("winningPlan"))

if MONGO_URI != "":
    explain_find("Cleaned_Orders", {"pickup_zone": "Central", "service_type": "Parcel"})

Collection: Cleaned_Orders
Filter: {'pickup_zone': 'Central', 'service_type': 'Parcel'}
Documents returned: 55
Documents examined: 55
Execution time ms: 0
Winning plan:
{'isCached': False, 'stage': 'FETCH', 'inputStage': {'stage': 'IXSCAN', 'keyPattern': {'pickup_zone': 1, 'service_type': 1}, 'indexName': 'idx_zone_service', 'isMultiKey': False, 'multiKeyPaths': {'pickup_zone': [], 'service_type': []}, 'isUnique': False, 'isSparse': False, 'isPartial': False, 'indexVersion': 2, 'direction': 'forward', 'indexBounds': {'pickup_zone': ['["Central", "Central"]'], 'service_type': ['["Parcel", "Parcel"]']}}}


## Interpreting my query optimisation result

I tested the query using MongoDB's `explain` command to check whether the index was used.

The query filtered orders by:

- `pickup_zone`;
- `service_type`.

I created the compound index:

`idx_zone_service`

The explain output should show an indexed query plan such as `IXSCAN`. This supports query optimisation because MongoDB can use the index rather than scanning every document in the collection.


# Final Summary of My MongoDB Workflow

This notebook shows how I developed the MongoDB part of the NorthStar database solution.

The completed workflow includes:

- connecting Google Colab to MongoDB Atlas using PyMongo;
- loading cleaned JSON files into MongoDB Atlas;
- creating separate collections for each cleaned dataset;
- remodelling selected datasets into the integrated `NorthStar_Service_Cases` collection;
- demonstrating Create, Read, Update, and Delete operations;
- running aggregation pipelines for NorthStar-style analysis;
- using `$lookup` to connect related collections;
- creating indexes for common query fields;
- using `explain` to check query performance and index usage.

This provides evidence for MongoDB development, NoSQL data modelling, CRUD operations, aggregation, indexing, and query optimisation.
